In [101]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

INPUT_PATH = "./ohchr_instruments_detailed.csv"
OUTPUTPATH = "./ohchr_instruments_detailed-instit.csv"

In [102]:
# Read dataframe
df = pd.read_csv(INPUT_PATH)

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def extract_institution(text):
    """
    Extract the line immediately following 'BY'.
    """
    if pd.isna(text):
        return None

    match = re.search(
        r"(?im)^BY\s*\n([^\n\r]+)",
        str(text)
    )

    return match.group(1).strip() if match else None


def extract_adoption_date(text):
    """
    Extract the line immediately following 'ADOPTED'.
    """
    if pd.isna(text):
        return None

    match = re.search(
        r"(?im)^ADOPTED\s*\n([^\n\r]+)",
        str(text)
    )

    return match.group(1).strip() if match else None


# --------------------------------------------------
# Create new columns
# --------------------------------------------------

df["institution"] = df["content"].apply(extract_institution)
# df["adoption_date"] = df["content"].apply(extract_adoption_date) # Not necessary because the 'adoption_by' already captures this
df.rename(columns={"adopted_by": "adoption_date"}, inplace=True)
df["resolution"] = ""
df["event"] = ""

# --------------------------------------------------
# Corrections
# --------------------------------------------------
standard_rules = 'https://www.ohchr.org/en/instruments-mechanisms/instruments/standard-rules-equalization-opportunities-persons-disabilities'
df.loc[df["url"] == standard_rules, "institution"] = "General Assembly"
df.loc[df["url"] == standard_rules, "resolution"] = "A/RES/48/96"

slavery_convention = "https://www.ohchr.org/en/instruments-mechanisms/instruments/slavery-convention"
df.loc[df["url"] == slavery_convention, "institution"] = "League of Nations"
df.loc[df["url"] == slavery_convention, "resolution"] = "LoN-1414"

stature_yugoslavia = "https://www.ohchr.org/en/instruments-mechanisms/instruments/statute-international-tribunal-prosecution-persons-responsible"
df.loc[df["url"] == stature_yugoslavia, "institution"] = "Security Council"
df.loc[df["url"] == stature_yugoslavia, "resolution"] = "S/RES/827(1993)"

to_correct = df[~df["url"].isin([standard_rules, slavery_convention, stature_yugoslavia])]

In [103]:
print(df[["title", "institution", "adoption_date"]].head())

                                                                                                             title  \
0                         Convention against Torture and Other Cruel, Inhuman or Degrading Treatment or Punishment   
1                                                            Convention on the Rights of Persons with Disabilities   
2                                   Optional Protocol to the Convention on the Rights of Persons with Disabilities   
3                     Optional Protocol to the Convention on the Rights of the Child on a communications procedure   
4  Optional Protocol to the Convention on the Rights of the Child on the involvement of children in armed conflict   

                                                                                 institution  \
0                                                          General Assembly resolution 39/46   
1                     Sixty-first session of the General Assembly by resolution A/RES/61/106   
2  

In [113]:
result = df[~df["institution"].str.contains("General Assembly", case=False, na=False)]
print(result[["url", "institution"]])

                                                                                                                             url  \
6                     https://www.ohchr.org/en/instruments-mechanisms/instruments/abolition-forced-labour-convention-1957-no-105   
9                            https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-independence-judiciary   
10                                     https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-role-lawyers   
11           https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-use-force-and-firearms-law-enforcement   
14                       https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-against-discrimination-education   
23                      https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-stateless-persons   
26                             https://www.ohchr.org/en/instruments-mechanis

In [112]:
# known events
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = ~df["institution"].str.contains("General Assembly", case=False, na=False)

mask_keys = df["institution"].str.contains("resolution", case=False, na=False)

result = df[mask_general_assembly & mask_keys]

print(result[["url", "institution"]])


                                                                                                                          url  \
23                   https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-stateless-persons   
50             https://www.ohchr.org/en/instruments-mechanisms/instruments/guidelines-action-children-criminal-justice-system   
68  https://www.ohchr.org/en/instruments-mechanisms/instruments/principles-effective-prevention-and-investigation-extra-legal   
81   https://www.ohchr.org/en/instruments-mechanisms/instruments/safeguards-guaranteeing-protection-rights-those-facing-death   
85    https://www.ohchr.org/en/instruments-mechanisms/instruments/statute-international-criminal-tribunal-prosecution-persons   
87     https://www.ohchr.org/en/instruments-mechanisms/instruments/supplementary-convention-abolition-slavery-slave-trade-and   

                                                                                                

In [106]:
# known events
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = df["institution"].str.contains("General Assembly", case=False, na=False)

mask_keys = df["institution"].str.contains("|".join(keys), case=False, na=False)

result = df.loc[mask_general_assembly & mask_keys, "institution"].to_list()

result


['Conference of Plenipotentiaries which met in 1959 and reconvened in 1961 in pursuance of General Assembly resolution 896 (IX)',
 'United Nations Conference of Plenipotentiaries on the Status of Refugees and Stateless Persons convened under General Assembly resolution 429 (V) of 14 December 1950',
 'the World Food Conference convened under General Assembly resolution 3180 (XXVIII) of 17 December 1973; and endorsed by General Assembly resolution 3348 (XXIX) of 17 December 1974']

In [107]:
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = df["institution"].str.contains("General Assembly", case=False, na=False)

mask_not_keys = ~df["institution"].str.contains("|".join(keys), case=False, na=False)

result = df.loc[mask_general_assembly & mask_not_keys, "institution"].to_list()

result

['General Assembly resolution 39/46',
 'Sixty-first session of the General Assembly by resolution A/RES/61/106',
 'the sixty-first session of the United Nations General Assembly by resolution A/RES/61/106',
 'General Assembly resolution A/RES/66/138',
 'General Assembly resolution A/RES/54/263',
 'sixty-third session of the General Assembly by resolution A/RES/63/117',
 'General Assembly resolution 60/147',
 'General Assembly resolution 45/111',
 'General Assembly resolution 43/173',
 'General Assembly resolution 34/169',
 'General Assembly resolution 317 (IV)',
 'General Assembly resolution 1763 A (XVII)',
 'United Nations General Assembly resolution 34/180',
 'General Assembly resolution 2391 (XXIII)',
 'General Assembly resolution 260 A (III)',
 'General Assembly resolution 44/25',
 'General Assembly resolution 40/34',
 'General Assembly resolution S-26/2',
 'General Assembly resolution 2542 (XXIV)',
 'General Assembly resolution 36/55',
 'General Assembly resolution 48/104',
 'Gene